# Metabo-Diet: harmonizing public diet and exercise metabolomics

**Audience.** Intermediate learners who know basic tabular data handling and have seen PCA, but do not need prior experience with Metabolomics Workbench (MW) or RefMet.

**Prerequisites.** Python 3.11 or 3.12 and the packages in `requirements-dev.txt`. The optional R companion uses R 4.3 or later. All learner data are public Metabolomics Workbench records.

**By the end, you can:**

1. retrieve and validate MW's split `summary`, `factors`, `analysis`, `metabolites`, and `data` endpoints;
2. derive participant IDs and tidy longitudinal factors without inventing a balanced panel;
3. distinguish repository-provided RefMet mappings from analytical equivalence;
4. audit isotope-labeled/internal-standard collisions before reporting biological overlap;
5. run log-transformed, median-imputed, autoscaled PCA within one study and analysis; and
6. state interpretation limits created by plasma-versus-serum, platform, timepoint, and co-intervention differences.

**Guided-analysis time:** about 40 minutes. The full asynchronous module is 2.5 hours.

## How to use the notebook with the learner guide

Complete the pretest in the learner guide before Lesson 1. For each lesson, use this sequence:

1. **READ** the named learner-guide lesson and its scientific guardrails.
2. **FIND** the matching notebook heading below by its stable key (`NB-L1` through `NB-L5`).
3. **RUN** only the cells labeled **Run now**, from top to bottom.
4. **OBSERVE** the named output; do not continue if a stop condition fails.
5. **RECORD** the requested answer in the learner worksheet or an editable notebook scaffold.
6. **CHECK** the completion statement before returning to the guide.

| Learner guide | Notebook section | Main notebook action |
|---|---|---|
| Lesson 1 - Why harmonization matters | `NB-L1` | Verify the environment, configuration, and scientific boundary |
| Lesson 2 - Comparing study design | `NB-L2` | Retrieve endpoints and audit study/sample structure |
| Lesson 3 - Harmonizing metadata/metabolites | `NB-L3` | Build and inspect the RefMet crosswalk |
| Lesson 4 - Guided analysis and interpretation | `NB-L4` | Summarize classes, run separate PCAs, and complete the class exercise |
| Lesson 5 - Access patterns and transfer | `NB-L5` | Draft a transfer decision and run the reproducibility audit |

The exact filename is `module/notebooks/metabo_diet_harmonization.ipynb`. Headings and visible keys, rather than cell numbers, are the durable cross-references used by the PDF guide.

<a id="nb-setup"></a>
## Environment setup (`NB-SETUP`) - complete before Lesson 1

Keep the extracted `module/` tree intact. Open a terminal in the directory that contains `module/`.

### 1. Install the runtimes once

- Install **Python 3.11 or 3.12** from <https://www.python.org/downloads/>. During Windows installation, enable the option that adds Python to `PATH`.
- Install **R 4.3 or later** from <https://cran.r-project.org/> only if you plan to use the R companion.
- A Jupyter interface is included in `requirements-dev.txt`; VS Code with the Jupyter extension is also acceptable.

### 2. Create the Python environment and install packages

macOS or Linux:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r module/notebooks/requirements-dev.txt
jupyter lab module/notebooks/metabo_diet_harmonization.ipynb
```

Windows PowerShell:

```powershell
py -3.12 -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install -r module/notebooks/requirements-dev.txt
jupyter lab module/notebooks/metabo_diet_harmonization.ipynb
```

Select the **Python 3 (Metabo-Diet)** kernel if prompted. For a deterministic noninteractive test, run `.venv/bin/python module/scripts/execute_notebook.py` on macOS/Linux or `.venv\Scripts\python module\scripts\execute_notebook.py` on Windows.

### 3. Install R packages if using the R companion

```bash
Rscript module/notebooks/install_r_packages.R
Rscript -e 'rmarkdown::render("module/notebooks/metabo_diet_R_appendix.Rmd")'
```

The cached path needs no network after package installation. Live retrieval is optional and is enabled by setting `METABO_DIET_LIVE=1` before starting Jupyter or R.

### Run now - verify Python and the complete package set

Run this diagnostic before importing the analysis libraries. **Stop** if it reports a missing package or an unsupported Python version; activate the intended environment and reinstall `requirements-dev.txt` before continuing.

In [1]:
from importlib.metadata import PackageNotFoundError, version
import platform
import sys

SUPPORTED_PYTHON = {(3, 11), (3, 12)}
REQUIRED_DISTRIBUTIONS = (
    "numpy",
    "pandas",
    "requests",
    "scikit-learn",
    "matplotlib",
    "IPython",
    "nbformat",
    "nbclient",
    "ipykernel",
    "jupyterlab",
)

if sys.version_info[:2] not in SUPPORTED_PYTHON:
    raise RuntimeError(
        f"Use Python 3.11 or 3.12; this kernel is {platform.python_version()}."
    )

installed = {}
missing = []
for distribution in REQUIRED_DISTRIBUTIONS:
    try:
        installed[distribution] = version(distribution)
    except PackageNotFoundError:
        missing.append(distribution)

if missing:
    raise RuntimeError(
        "Missing packages: " + ", ".join(missing)
        + ". Install module/notebooks/requirements-dev.txt and restart the kernel."
    )

print(f"Python {platform.python_version()} - environment check passed")
for distribution, installed_version in installed.items():
    print(f"  {distribution}=={installed_version}")

Python 3.12.8 - environment check passed
  numpy==2.2.6
  pandas==2.2.3
  requests==2.32.5
  scikit-learn==1.7.1
  matplotlib==3.10.5
  IPython==9.15.0
  nbformat==5.10.4
  nbclient==0.10.2
  ipykernel==6.30.1
  jupyterlab==4.6.2


<a id="nb-l1"></a>
## Lesson 1 - Why harmonization matters (`NB-L1`)

**Guide cross-reference.** Read learner-guide Lesson 1 before running the configuration cell. Then:

1. Review the scientific boundary below.
2. Run the configuration cell once.
3. Confirm the two accessions and the retrieval mode.
4. Record why the two quantitative matrices must remain separate.

### Scientific boundary before any analysis

The studies answer different questions. ST001521 is longitudinal plasma metabolomics during controlled feeding, with antibiotics on days 6–8 and a polyethylene glycol purge on day 7. ST003348 is serum metabolomics around an acute endurance race-walking bout. Exact RefMet names let us compare *nomenclature and coverage*; they do not prove matching isomers, annotation certainty, extraction recovery, or quantitative calibration.

For that reason, this tutorial compares metadata, mapped-name presence, class coverage, and separately standardized within-study patterns. It never concatenates the raw peak-area matrices.

### Run now - configure paths, packages, and locked accessions

Expected output: the local `module/` directory and either `validated cache` (default) or `live with cache fallback`. **Stop** if either accession differs from `ST001521` and `ST003348`.

In [2]:
from __future__ import annotations

import hashlib
import json
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def locate_module_dir(start: Path) -> Path:
    for parent in [start.resolve(), *start.resolve().parents]:
        if (parent / "data" / "raw").is_dir() and (parent / "scripts").is_dir():
            return parent
        if (parent / "module" / "data" / "raw").is_dir():
            return parent / "module"
    raise FileNotFoundError("Could not locate module/data/raw")


MODULE_DIR = locate_module_dir(Path.cwd())
RAW_DIR = MODULE_DIR / "data" / "raw"
DERIVED_DIR = MODULE_DIR / "data" / "derived"
FIGURES_DIR = MODULE_DIR / "figures"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Keep plotting caches out of learner artifacts while remaining sandbox-friendly.
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "metabo_diet_mpl"))
sys.path.insert(0, str(MODULE_DIR / "scripts"))

from metabo_diet_pipeline import (
    MW_ENDPOINTS,
    build_metabolite_crosswalk,
    derive_tidy_factors,
    load_studies,
    pca_summary_frame,
    plot_class_summary,
    plot_diet_pca,
    plot_exercise_pca,
    plot_overlap_counts,
    run_within_study_pca,
    write_pca_outputs,
)

# Required accession constants: change these only after re-auditing study design.
DIET_ACCESSION = "ST001521"
EXERCISE_ACCESSION = "ST003348"

# Matched positive-ion, reversed-phase analyses chosen for readable demonstrations.
# Each is analyzed on its own scale; this selection does not make them quantitatively equivalent.
DIET_ANALYSIS_ID = "AN002534"
EXERCISE_ANALYSIS_ID = "AN005483"

# Set METABO_DIET_LIVE=1 to try each split REST endpoint first. The validated cache
# is the deterministic default and is also used whenever a live request fails.
PREFER_LIVE_API = os.getenv("METABO_DIET_LIVE", "0") == "1"

pd.set_option("display.max_colwidth", 80)
print("Module directory located successfully: module/")
print(f"Retrieval mode: {'live with cache fallback' if PREFER_LIVE_API else 'validated cache'}")

Module directory located successfully: module/
Retrieval mode: validated cache


<a id="nb-l2"></a>
## Lesson 2 - Comparing study design and phenotype capture (`NB-L2`)

**Guide cross-reference.** Read learner-guide Lesson 2 and open the cohort-comparison worksheet. In this section:

1. Load all five split endpoints for each study.
2. Verify endpoint and analysis counts.
3. Derive biological samples, participant IDs, specimen, condition, and time.
4. Record at least one direct, partial, and non-comparable field in the worksheet.

### L2.1 Retrieve and validate split MW endpoints

MW publishes different record types at separate URLs. `load_studies` requests each endpoint independently when live mode is enabled, validates required fields and study IDs, and then falls back endpoint-by-endpoint to `data/raw/{accession}_{endpoint}.json` if needed. This is safer than treating a partial or malformed response as complete data.

### Run now - load the ten endpoint responses

Expected output: ten rows, one for each study-endpoint pair. **Stop** if any source is neither `validated cache` nor a recorded live response with cache fallback.

In [3]:
studies, source_log = load_studies(
    [DIET_ACCESSION, EXERCISE_ACCESSION],
    RAW_DIR,
    prefer_live=PREFER_LIVE_API,
)
source_log_path = DERIVED_DIR / "input_source_log.csv"
source_log.to_csv(source_log_path, index=False)

assert set(source_log.endpoint) == set(MW_ENDPOINTS)
assert source_log.shape[0] == 10
display(source_log[["study_id", "endpoint", "source", "url"]])

,study_id,endpoint,source,url
0,ST001521,summary,validated cache,https://www.metabolomicsworkbench.org/rest/study/study_id/ST001521/summary
1,ST001521,factors,validated cache,https://www.metabolomicsworkbench.org/rest/study/study_id/ST001521/factors
2,ST001521,analysis,validated cache,https://www.metabolomicsworkbench.org/rest/study/study_id/ST001521/analysis
3,ST001521,metabolites,validated cache,https://www.metabolomicsworkbench.org/rest/study/study_id/ST001521/metabolites
4,ST001521,data,validated cache,https://www.metabolomicsworkbench.org/rest/study/study_id/ST001521/data
5,ST003348,summary,validated cache,https://www.metabolomicsworkbench.org/rest/study/study_id/ST003348/summary
6,ST003348,factors,validated cache,https://www.metabolomicsworkbench.org/rest/study/study_id/ST003348/factors
7,ST003348,analysis,validated cache,https://www.metabolomicsworkbench.org/rest/study/study_id/ST003348/analysis
8,ST003348,metabolites,validated cache,https://www.metabolomicsworkbench.org/rest/study/study_id/ST003348/metabolites
9,ST003348,data,validated cache,https://www.metabolomicsworkbench.org/rest/study/study_id/ST003348/data


### Run now - compare endpoint sizes and analytical modes

Observe the factor, metabolite, and analysis counts. Record the analysis IDs, modes, and units in the cohort-comparison worksheet before continuing.

In [4]:
endpoint_counts = []
for study_id, payloads in studies.items():
    for endpoint, payload in payloads.items():
        endpoint_counts.append(
            {
                "study_id": study_id,
                "endpoint": endpoint,
                "records": 1 if endpoint == "summary" else len(payload),
            }
        )
endpoint_counts = pd.DataFrame(endpoint_counts)
display(endpoint_counts.pivot(index="endpoint", columns="study_id", values="records"))

analysis_table = pd.concat(
    [pd.DataFrame(payloads["analysis"].values()) for payloads in studies.values()],
    ignore_index=True,
)
display(
    analysis_table[
        ["study_id", "analysis_id", "analysis_summary", "chromatography_type", "ion_mode", "units"]
    ]
)

study_id,ST001521,ST003348
endpoint,,
analysis,4,2
data,567,593
factors,160,76
metabolites,567,593
summary,1,1


,study_id,analysis_id,analysis_summary,chromatography_type,ion_mode,units
0,ST001521,AN002533,HILIC POSITIVE ION MODE,HILIC,POSITIVE,unitless peak areas
1,ST001521,AN002534,Reversed phase POSITIVE ION MODE,Reversed phase,POSITIVE,unitless peak areas
2,ST001521,AN002535,HILIC NEGATIVE ION MODE,HILIC,NEGATIVE,unitless peak areas
3,ST001521,AN002536,Reversed phase NEGATIVE ION MODE,Reversed phase,NEGATIVE,unitless peak areas
4,ST003348,AN005483,Reversed phase POSITIVE ION MODE,Reversed phase,POSITIVE,Peak area
5,ST003348,AN005484,Reversed phase NEGATIVE ION MODE,Reversed phase,NEGATIVE,Peak area


**Observe and record.** The expected endpoint sizes are 160 versus 76 factor rows, 567 versus 593 analyte-analysis rows, and four versus two analyses. Counts describe the endpoint actually used - not larger project-level totals that may appear in a paper or narrative. If these values differ, record the discrepancy and stop to investigate the release/cache version.

### L2.2 Tidy factors and derive participant IDs

Factor strings are parsed at the first colon in each pipe-separated component. The rules are explicit:

- ST001521 participant ID = digits before the first hyphen; pooled-plasma QC IDs `QPP01`–`QPP10` are excluded.
- ST003348 participant ID = integer before the underscore. Suffixes 1–4 must agree with `rest`, `stat`, `rec3`, and `rec22`.
- Original source terms remain in the table; a separate field harmonizes plasma versus serum.

### Run now - create the biological sample table

Expected output: 150 diet-study biological samples from 30 participants and 76 exercise-study samples from 19 participants. The displayed rows retain original labels beside harmonized fields.

In [5]:
tidy_factors = derive_tidy_factors(
    studies, DIET_ACCESSION, EXERCISE_ACCESSION
)
tidy_factors_path = DERIVED_DIR / "tidy_factors.csv"
tidy_factors.to_csv(tidy_factors_path, index=False)

factor_counts = (
    tidy_factors.groupby(["study_id", "study_role"], as_index=False)
    .agg(
        samples=("local_sample_id", "nunique"),
        participants=("participant_id", "nunique"),
        timepoints=("time_original", "nunique"),
    )
)
display(factor_counts)
display(tidy_factors.head(8))

,study_id,study_role,samples,participants,timepoints
0,ST001521,diet,150,30,5
1,ST003348,exercise,76,19,4


,study_id,study_role,local_sample_id,mb_sample_id,participant_id,participant_id_rule,factor_string_original,sample_source_original,specimen_harmonized,condition_original,sex_original,time_original,time_harmonized,time_order,trajectory_role,interpretation_context,factor_parse_status
0,ST001521,diet,9002-3-PE,SA128095,9002,digits before first hyphen,Study_Diet:Vegan | Sex:Male | Time:Baseline,Blood (plasma),plasma,Vegan,Male,Baseline,baseline,0,baseline,Pre-intervention,parsed_and_validated
1,ST001521,diet,9002-3-PD,SA128098,9002,digits before first hyphen,Study_Diet:Vegan | Sex:Male | Time:Day 5,Blood (plasma),plasma,Vegan,Male,Day 5,day_05,5,diet_exposure,Diet exposure before antibiotics,parsed_and_validated
2,ST001521,diet,9002-3-PC,SA128099,9002,digits before first hyphen,Study_Diet:Vegan | Sex:Male | Time:Day 9,Blood (plasma),plasma,Vegan,Male,Day 9,day_09,9,diet_plus_microbiome_perturbation,Antibiotics days 6-8 and PEG purge day 7; not a diet-only contrast,parsed_and_validated
3,ST001521,diet,9002-3-PB,SA128096,9002,digits before first hyphen,Study_Diet:Vegan | Sex:Male | Time:Day 12,Blood (plasma),plasma,Vegan,Male,Day 12,day_12,12,diet_plus_microbiome_perturbation,Post-antibiotics/PEG interval; not a diet-only contrast,parsed_and_validated
4,ST001521,diet,9002-3-PA,SA128097,9002,digits before first hyphen,Study_Diet:Vegan | Sex:Male | Time:Day 15,Blood (plasma),plasma,Vegan,Male,Day 15,day_15,15,diet_plus_microbiome_perturbation,Post-antibiotics/PEG interval; not a diet-only contrast,parsed_and_validated
5,ST001521,diet,9003-2-PE,SA128035,9003,digits before first hyphen,Study_Diet:Modulen | Sex:Male | Time:Baseline,Blood (plasma),plasma,Modulen,Male,Baseline,baseline,0,baseline,Pre-intervention,parsed_and_validated
6,ST001521,diet,9003-2-PD,SA128038,9003,digits before first hyphen,Study_Diet:Modulen | Sex:Male | Time:Day 5,Blood (plasma),plasma,Modulen,Male,Day 5,day_05,5,diet_exposure,Diet exposure before antibiotics,parsed_and_validated
7,ST001521,diet,9003-2-PC,SA128039,9003,digits before first hyphen,Study_Diet:Modulen | Sex:Male | Time:Day 9,Blood (plasma),plasma,Modulen,Male,Day 9,day_09,9,diet_plus_microbiome_perturbation,Antibiotics days 6-8 and PEG purge day 7; not a diet-only contrast,parsed_and_validated


### Run now - audit timepoints and stop conditions

The assertions verify exclusion of `QPP...` pooled-QC samples from biological counts and the expected study-specific sample totals. **Stop** on any assertion failure; do not repair counts by dropping rows without evidence.

In [6]:
timepoint_counts = (
    tidy_factors.groupby(
        ["study_id", "condition_original", "time_original", "time_order"],
        dropna=False,
        as_index=False,
    )
    .agg(samples=("local_sample_id", "nunique"))
    .sort_values(["study_id", "condition_original", "time_order"])
)
display(timepoint_counts)

assert tidy_factors.query("study_id == @DIET_ACCESSION").local_sample_id.str.startswith("QPP").sum() == 0
assert tidy_factors.query("study_id == @DIET_ACCESSION").shape[0] == 150
assert tidy_factors.query("study_id == @EXERCISE_ACCESSION").shape[0] == 76

,study_id,condition_original,time_original,time_order,samples
0,ST001521,Modulen,Baseline,0,10
3,ST001521,Modulen,Day 5,5,10
4,ST001521,Modulen,Day 9,9,10
1,ST001521,Modulen,Day 12,12,10
2,ST001521,Modulen,Day 15,15,10
5,ST001521,Vegan,Baseline,0,10
8,ST001521,Vegan,Day 5,5,10
9,ST001521,Vegan,Day 9,9,10
6,ST001521,Vegan,Day 12,12,10
7,ST001521,Vegan,Day 15,15,10


### Learner edit - inspect one study before comparing it

Choose `DIET_ACCESSION` or `EXERCISE_ACCESSION`, predict the number of participants and timepoints, then run the scaffold. Record one design difference that makes a pooled quantitative comparison unsafe.

In [7]:
# Learner-edit cell: change this to EXERCISE_ACCESSION for the second case.
STUDY_TO_AUDIT = DIET_ACCESSION

study_audit = (
    tidy_factors.query("study_id == @STUDY_TO_AUDIT")
    .groupby(["condition_original", "time_original"], dropna=False, as_index=False)
    .agg(samples=("local_sample_id", "nunique"), participants=("participant_id", "nunique"))
)
display(study_audit)

,condition_original,time_original,samples,participants
0,Modulen,Baseline,10,10
1,Modulen,Day 12,10,10
2,Modulen,Day 15,10,10
3,Modulen,Day 5,10,10
4,Modulen,Day 9,10,10
5,Vegan,Baseline,10,10
6,Vegan,Day 12,10,10
7,Vegan,Day 15,10,10
8,Vegan,Day 5,10,10
9,Vegan,Day 9,10,10


### Interpretation guardrail: a time label is not a treatment label

Do not force a balanced panel: the FARMM factor endpoint has nine Western-male Day 5 rows and eleven Western-male Day 9 rows. More importantly, days 9, 12, and 15 occur during or after antibiotic/PEG perturbation, so they are not diet-only contrasts. Exercise collections also differ in fasting status and clock time. `interpretation_context` keeps those design facts beside every sample.

**Lesson 2 complete when:** the endpoint audit passes and your worksheet distinguishes samples from participants, plasma from serum, and diet-study days from exercise-recovery hours.

<a id="nb-l3"></a>
## Lesson 3 - Harmonizing metabolomics and metadata (`NB-L3`)

**Guide cross-reference.** Read learner-guide Lesson 3 and open the metabolite/metadata crosswalk worksheet. In this section:

1. Preserve source names and analysis identifiers.
2. Build the raw exact-name overlap.
3. Audit labeled-standard collisions before calling an overlap biological.
4. Trace one retained RefMet name back to both studies.

### L3.1 Build an auditable RefMet crosswalk (`NB-L3-CROSSWALK`)

The pipeline preserves every source-reported name, analysis ID, MW metabolite ID, RefMet name/class, mapping evidence, confidence language, and inclusion decision.

The raw exact intersection is intentionally shown before cleanup. ST003348 contains ten explicit stable-isotope/internal-standard rows. Eight RefMet labels from those rows collide with the raw overlap. We conservatively remove those labels - even when an unlabeled row shares the same RefMet label - because the study-level endpoint alone does not tell us which signal should represent the biological compound. This produces the pre-specified conservative overlap of 145.

### Run now - construct and save the crosswalk audit tables

Expected output: 510 diet names, 475 exercise names, 153 raw exact overlaps, and 145 conservatively retained overlaps. **Stop** if the output skips the raw audit stage or drops provenance columns.

In [8]:
with (RAW_DIR / "refmet_classification.json").open(encoding="utf-8") as handle:
    refmet_classification = json.load(handle)

(
    mapping_audit,
    overlap_audit,
    metabolite_crosswalk,
    class_summary,
    overlap_counts,
) = build_metabolite_crosswalk(
    studies,
    refmet_classification,
    DIET_ACCESSION,
    EXERCISE_ACCESSION,
)

derived_tables = {
    "metabolite_mapping_audit.csv": mapping_audit,
    "refmet_overlap_audit.csv": overlap_audit,
    "metabolite_crosswalk.csv": metabolite_crosswalk,
    "refmet_class_summary.csv": class_summary,
    "refmet_overlap_counts.csv": pd.DataFrame(
        [{"metric": key, "value": value} for key, value in overlap_counts.items()]
    ),
}
for filename, table in derived_tables.items():
    table.to_csv(DERIVED_DIR / filename, index=False)

display(pd.DataFrame([overlap_counts]).T.rename(columns={0: "count"}))

,count
diet_unique_nonblank_refmet,510
exercise_unique_nonblank_refmet,475
raw_exact_refmet_overlap,153
exercise_isotope_internal_standard_rows,10
raw_overlap_labels_with_standard_collision,8
conservative_biological_refmet_overlap,145
conservative_overlap_with_refmet_class,145


### Run now - inspect the excluded labeled-standard evidence

Observe the source-reported label, RefMet label, detection evidence, and row decision. The assertions require 153 raw rows, eight standard-label collisions, and 145 retained names.

In [9]:
standard_audit = mapping_audit.query(
    "study_id == @EXERCISE_ACCESSION and isotope_internal_standard_row"
)[
    [
        "analysis_id",
        "source_reported_name",
        "refmet_name",
        "mapping_status",
        "standard_detection_evidence",
        "row_decision",
    ]
]
display(standard_audit)

assert overlap_audit.shape[0] == 153
assert overlap_audit.exercise_standard_collision.sum() == 8
assert metabolite_crosswalk.shape[0] == 145

,analysis_id,source_reported_name,refmet_name,mapping_status,standard_detection_evidence,row_decision
596,AN005483,AcCa(12:0)-D9,CAR 12:0,excluded_isotope_labeled_or_internal_standard,Explicit stable-isotope suffix in source-reported name,Exclude row from biological analyses
597,AN005483,AcCa(18:0)-D3,CAR 18:0,excluded_isotope_labeled_or_internal_standard,Explicit stable-isotope suffix in source-reported name,Exclude row from biological analyses
618,AN005483,Clenbuterol-D9,,excluded_isotope_labeled_or_internal_standard,Explicit stable-isotope suffix in source-reported name,Exclude row from biological analyses
654,AN005483,Hippuric acid-D5,Hippuric acid,excluded_isotope_labeled_or_internal_standard,Explicit stable-isotope suffix in source-reported name,Exclude row from biological analyses
709,AN005483,Lysine-d4,Standard,excluded_isotope_labeled_or_internal_standard,Explicit stable-isotope suffix in source-reported name,Exclude row from biological analyses
852,AN005483,Taurine-D4,Taurine,excluded_isotope_labeled_or_internal_standard,Explicit stable-isotope suffix in source-reported name,Exclude row from biological analyses
976,AN005484,CDCA-D4,Chenodeoxycholic acid,excluded_isotope_labeled_or_internal_standard,Explicit stable-isotope suffix in source-reported name,Exclude row from biological analyses
980,AN005484,Chloramphenicol-D5,Chloramphenicol,excluded_isotope_labeled_or_internal_standard,Explicit stable-isotope suffix in source-reported name,Exclude row from biological analyses
1110,AN005484,Palmitic acid-[13C]16,Palmitic acid,excluded_isotope_labeled_or_internal_standard,Explicit stable-isotope suffix in source-reported name,Exclude row from biological analyses
1130,AN005484,Stearic acid-D35,Stearic acid,excluded_isotope_labeled_or_internal_standard,Explicit stable-isotope suffix in source-reported name,Exclude row from biological analyses


### Run now - plot the overlap stages and preview retained provenance

The bar chart must distinguish the raw and conservative counts. The table underneath should let you trace a retained name to both source labels and analysis IDs.

In [10]:
plot_overlap_counts(overlap_counts, FIGURES_DIR / "refmet_overlap_summary.png")
display(
    metabolite_crosswalk[
        [
            "refmet_name",
            "main_class",
            "diet_source_reported_names",
            "diet_analysis_ids",
            "exercise_source_reported_names",
            "exercise_analysis_ids",
            "mapping_confidence",
            "decision_reason",
        ]
    ].head(10)
)

,refmet_name,main_class,diet_source_reported_names,diet_analysis_ids,exercise_source_reported_names,exercise_analysis_ids,mapping_confidence,decision_reason
0,"1,7-Dimethyluric acid",Purines,"1,7-Dimethyluric acid",AN002533; AN002536,"1,7-Dimethyluric acid",AN005484,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision
1,1-Methyl nicotinamide,Pyridine alkaloids,1-Methyl nicotinamide,AN002533,1-Methylnicotinamide,AN005483,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision
2,1-Methyladenosine,Purines,1-Methyladenosine,AN002533,1-Methyladenosine,AN005483,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision
3,"12,13-DiHOME",Octadecanoids,"12,13-DiHOME",AN002536,"12,13-DHOME",AN005484,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision
4,2-Aminocaprylic acid,Fatty acids,2-Aminooctanoic acid,AN002533; AN002536,2-Aminooctanoic acid,AN005483,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision
5,2-Hydroxyglutaric acid,Fatty acids,2-Hydroxyglutaric acid,AN002535,2-Hydroxyglutaric acid,AN005484,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision
6,2-Hydroxymyristic acid,Fatty acids,2-Hydroxymyristic acid,AN002536,2-Hydroxymyristic acid(FFA(14:0-OH),AN005484,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision
7,2-Methylguanosine,Purines,2-Methylguanosine,AN002533,2-Methylguanosine,AN005483,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision
8,2-Oxoglutaric acid,TCA acids,Oxoglutaric acid,AN002535,Oxoglutaric acid,AN005484,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision
9,3-Dehydroxycarnitine,Fatty esters,3-Dehydroxycarnitine,AN002533,4-Trimethylammoniobutanoic acid,AN005483,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision


![High-contrast bar chart showing 510 diet RefMet names, 475 exercise RefMet names, a raw exact overlap of 153, and a conservative biological overlap of 145 after standard-label cleanup.](../figures/refmet_overlap_summary.png)

**Interpretation.** The 145-name crosswalk is an auditable vocabulary bridge. It is not a list of quantitatively interchangeable measurements.

### Learner edit - trace one retained name

Choose a different row if desired. Record the submitted names and analysis IDs from both studies, then write one sentence explaining why the shared RefMet key does not prove equal concentration or identification certainty.

In [11]:
# Learner-edit cell: replace this value with another retained RefMet name.
REFMET_TO_TRACE = metabolite_crosswalk.iloc[0]["refmet_name"]

trace_columns = [
    "refmet_name",
    "main_class",
    "diet_source_reported_names",
    "diet_analysis_ids",
    "exercise_source_reported_names",
    "exercise_analysis_ids",
    "mapping_confidence",
    "decision_reason",
]
display(metabolite_crosswalk.query("refmet_name == @REFMET_TO_TRACE")[trace_columns])

,refmet_name,main_class,diet_source_reported_names,diet_analysis_ids,exercise_source_reported_names,exercise_analysis_ids,mapping_confidence,decision_reason
0,"1,7-Dimethyluric acid",Purines,"1,7-Dimethyluric acid",AN002533; AN002536,"1,7-Dimethyluric acid",AN005484,repository-provided RefMet mapping; nomenclature bridge only,Retain; no ST003348 isotope/internal-standard collision


**Lesson 3 complete when:** all three overlap assertions pass and your worksheet preserves source names, analysis IDs, mapping evidence, uncertainty, and the exclusion decision for at least one example.

<a id="nb-l4"></a>
## Lesson 4 - Guided analysis and biological interpretation (`NB-L4`)

**Guide cross-reference.** Read learner-guide Lesson 4 before generating figures. Use the `NB-L4-CLASS`, `NB-L4-PCA-DIET`, and `NB-L4-PCA-EXERCISE` keys cited in the guide.

### L4.1 Summarize overlap by RefMet class (`NB-L4-CLASS`)

Class counts come from the cached RefMet classification table using exact name lookup. The full CSV keeps both super-class and main-class labels so a broad class cannot silently replace a specific one.

### Run now - create the class summary

Expected output: the class counts sum to 145. Treat the result as assay/name coverage, not pathway enrichment.

In [12]:
plot_class_summary(class_summary, FIGURES_DIR / "refmet_class_summary.png")
display(class_summary.head(15))

assert class_summary.refmet_count.sum() == 145
assert overlap_counts["conservative_overlap_with_refmet_class"] == 145

,super_class,main_class,refmet_count
0,Organic acids,Amino acids and peptides,37
1,Fatty Acyls,Fatty acids,25
2,Fatty Acyls,Fatty esters,15
3,Nucleic acids,Purines,11
4,Sterol Lipids,Bile acids,8
5,Alkaloids,Pyridine alkaloids,7
6,Alkaloids,Tryptophan alkaloids,4
7,Nucleic acids,Pyrimidines,4
8,Organic acids,TCA acids,3
9,Sphingolipids,Sphingoid bases,3


![Horizontal high-contrast bar chart of the twelve largest RefMet main classes among the 145 conservative shared metabolite names.](../figures/refmet_class_summary.png)

Class abundance reflects what these assays annotated and what RefMet classified. It is not pathway enrichment and does not adjust for how many compounds exist in each class.

### L4.2 PCA - within each study and selected analysis only

For a transparent teaching workflow, each selected matrix is processed independently:

1. keep biological sample IDs from the validated factor table;
2. for ST003348, remove explicit isotope/internal-standard feature rows;
3. exclude features with more than 20% missing values;
4. transform as `log2(peak area + 1)`;
5. median-impute each feature on the logged scale;
6. remove zero-variance features and autoscale each remaining feature; and
7. fit a two-component PCA to that one study/analysis matrix.

PCA is exploratory. Separation can reflect biology, time, diet, fasting, collection, analytical mode, or other unmodeled structure. PCA does not establish differential abundance or causal effects.

### Run now - fit the diet-study PCA (`NB-L4-PCA-DIET`)

Expected output: 150 biological plasma samples from `AN002534`; pooled `QPP...` samples are excluded. Record PC1/PC2 explained variance and two plausible sources of pattern besides diet.

In [13]:
diet_pca = run_within_study_pca(
    DIET_ACCESSION,
    DIET_ANALYSIS_ID,
    studies[DIET_ACCESSION],
    tidy_factors,
    exclude_isotope_standards=False,
    max_missing_fraction=0.20,
)
write_pca_outputs(diet_pca, DERIVED_DIR)
plot_diet_pca(diet_pca, FIGURES_DIR / "ST001521_AN002534_pca.png")

display(pca_summary_frame([diet_pca]))

,study_id,analysis_id,samples,input_feature_rows,pca_features,missingness_threshold,transform,imputation,scaling,PC1_variance_percent,PC2_variance_percent
0,ST001521,AN002534,150,224,212,0.2,log2(peak area + 1),feature median after log2 transform,"feature autoscaling (mean 0, SD 1)",34.359138,11.623835


![ST001521 positive reversed-phase PCA with points colored by original diet factor and marker shapes indicating Baseline, Day 5, Day 9, Day 12, and Day 15; axes report explained variance.](../figures/ST001521_AN002534_pca.png)

**Read cautiously.** The plot summarizes 150 plasma samples from AN002534. It does not isolate a diet effect: arms differ in setting and prior diet, while post-day-5 collections are entangled with antibiotics and PEG. Overlapping points do not prove equivalence, and separated points do not identify the features or mechanisms responsible.

### Run now - fit the exercise-study PCA (`NB-L4-PCA-EXERCISE`)

Expected output: 76 serum samples from `AN005483`, after explicit labeled-standard exclusion. Record PC1/PC2 explained variance and why this is not a repeated-measures hypothesis test.

In [14]:
exercise_pca = run_within_study_pca(
    EXERCISE_ACCESSION,
    EXERCISE_ANALYSIS_ID,
    studies[EXERCISE_ACCESSION],
    tidy_factors,
    exclude_isotope_standards=True,
    max_missing_fraction=0.20,
)
write_pca_outputs(exercise_pca, DERIVED_DIR)
plot_exercise_pca(exercise_pca, FIGURES_DIR / "ST003348_AN005483_pca.png")

pca_summary = pca_summary_frame([diet_pca, exercise_pca])
pca_summary.to_csv(DERIVED_DIR / "pca_preprocessing_summary.csv", index=False)
display(pca_summary)

,study_id,analysis_id,samples,input_feature_rows,pca_features,missingness_threshold,transform,imputation,scaling,PC1_variance_percent,PC2_variance_percent
0,ST001521,AN002534,150,224,212,0.2,log2(peak area + 1),feature median after log2 transform,"feature autoscaling (mean 0, SD 1)",34.359138,11.623835
1,ST003348,AN005483,76,319,313,0.2,log2(peak area + 1),feature median after log2 transform,"feature autoscaling (mean 0, SD 1)",16.114680,8.417490


![ST003348 positive reversed-phase PCA of 76 serum samples, with high-contrast colors for rest, immediate post-exercise, 3-hour recovery, and 22-hour recovery; axes report explained variance.](../figures/ST003348_AN005483_pca.png)

**Read cautiously.** The plot summarizes AN005483 only, after excluding its explicit labeled-standard rows. Within-person repeated measures remain correlated, and collection time overlaps with fasting and clock-time changes. PCA alone is not a repeated-measures test.

### Non-negotiable guardrail

There is intentionally no code that stacks ST001521 and ST003348 peak areas. Plasma versus serum, separate extraction and chromatography methods, different analytical analyses, and uncalibrated peak-area scales make a combined PCA dominated by study/platform effects and therefore uninterpretable as a diet-versus-exercise contrast.

Safe cross-study targets here are RefMet presence, class coverage, mapping provenance, and separately computed within-study summaries.

### L4.3 Learner edit - audit one shared class

Choose one RefMet main class and answer:

1. How many conservative shared names belong to it?
2. Do any names map to more than one analysis within either study?
3. What claim can you make, and what claim must you avoid?

Predict your result before running the scaffold. Change `CLASS_TO_INSPECT` to explore another class.

In [15]:
# Answer scaffold (runs with a safe default; edit the class name to explore).
CLASS_TO_INSPECT = class_summary.iloc[0]["main_class"]

class_crosswalk = metabolite_crosswalk.query("main_class == @CLASS_TO_INSPECT").copy()
class_answer = {
    "main_class": CLASS_TO_INSPECT,
    "shared_refmet_names": class_crosswalk.refmet_name.nunique(),
    "diet_names_with_multiple_analyses": int(
        class_crosswalk.diet_analysis_ids.str.contains(";").sum()
    ),
    "exercise_names_with_multiple_analyses": int(
        class_crosswalk.exercise_analysis_ids.str.contains(";").sum()
    ),
}
display(pd.DataFrame([class_answer]))
display(
    class_crosswalk[
        [
            "refmet_name",
            "diet_source_reported_names",
            "diet_analysis_ids",
            "exercise_source_reported_names",
            "exercise_analysis_ids",
        ]
    ].head(12)
)

,main_class,shared_refmet_names,diet_names_with_multiple_analyses,exercise_names_with_multiple_analyses
0,Amino acids and peptides,37,2,0


,refmet_name,diet_source_reported_names,diet_analysis_ids,exercise_source_reported_names,exercise_analysis_ids
13,4-Acetamidobutanoic acid,4-Acetamidobutanoic acid,AN002533,4-Acetamidobutanoic acid,AN005483
19,Alanine,Alanine,AN002533,Alanine+Sarcosine,AN005483
22,alpha-N-Phenylacetylglutamine,alpha-N-Phenylacetylglutamine,AN002533,Phenylacetylglutamine,AN005483
23,Aminoadipic acid,Aminoadipic acid,AN002535,Aminoadipic acid,AN005484
26,Arginine,Arginine,AN002533,L-Arginine,AN005483
27,Asparagine,Asparagine,AN002533,L-Asparagine,AN005483
28,Aspartic acid,Aspartic acid,AN002535,L-Aspartic acid,AN005483
30,Betaine,Betaine,AN002533,Betaine,AN005483
51,Citrulline,Citrulline,AN002533,Citrulline,AN005483
54,Creatine,Creatine,AN002533,Creatine,AN005483


### Learner edit - run one preprocessing sensitivity check

Predict whether a stricter missingness threshold will change the retained feature count or explained variance. Change `SENSITIVITY_MAX_MISSING` to a value from 0 through 1, run the cell, and record which observations persist. This is a within-study sensitivity check, not a license to tune the threshold for a preferred plot.

In [16]:
# Learner-edit cell: try 0.10, 0.15, or another justified threshold.
SENSITIVITY_STUDY = DIET_ACCESSION
SENSITIVITY_MAX_MISSING = 0.10

if not 0 <= SENSITIVITY_MAX_MISSING <= 1:
    raise ValueError("SENSITIVITY_MAX_MISSING must be between 0 and 1")

if SENSITIVITY_STUDY == DIET_ACCESSION:
    sensitivity_analysis_id = DIET_ANALYSIS_ID
    sensitivity_payload = studies[DIET_ACCESSION]
    sensitivity_baseline = diet_pca
    sensitivity_exclude_standards = False
elif SENSITIVITY_STUDY == EXERCISE_ACCESSION:
    sensitivity_analysis_id = EXERCISE_ANALYSIS_ID
    sensitivity_payload = studies[EXERCISE_ACCESSION]
    sensitivity_baseline = exercise_pca
    sensitivity_exclude_standards = True
else:
    raise ValueError("SENSITIVITY_STUDY must be DIET_ACCESSION or EXERCISE_ACCESSION")

sensitivity_pca = run_within_study_pca(
    SENSITIVITY_STUDY,
    sensitivity_analysis_id,
    sensitivity_payload,
    tidy_factors,
    exclude_isotope_standards=sensitivity_exclude_standards,
    max_missing_fraction=SENSITIVITY_MAX_MISSING,
)
sensitivity_comparison = pd.concat(
    [
        pca_summary_frame([sensitivity_baseline]).assign(run="primary threshold 0.20"),
        pca_summary_frame([sensitivity_pca]).assign(
            run=f"sensitivity threshold {SENSITIVITY_MAX_MISSING:.2f}"
        ),
    ],
    ignore_index=True,
)
display(sensitivity_comparison)

,study_id,analysis_id,samples,input_feature_rows,pca_features,missingness_threshold,transform,imputation,scaling,PC1_variance_percent,PC2_variance_percent,run
0,ST001521,AN002534,150,224,212,0.2,log2(peak area + 1),feature median after log2 transform,"feature autoscaling (mean 0, SD 1)",34.359138,11.623835,primary threshold 0.20
1,ST001521,AN002534,150,224,205,0.1,log2(peak area + 1),feature median after log2 transform,"feature autoscaling (mean 0, SD 1)",34.512952,11.790481,sensitivity threshold 0.10


### Sample answer

Reveal this only after writing your own answer:

“This class contains the reported number of exact, conservatively retained shared RefMet names. Some names may occur in multiple analysis modes, which is visible in the analysis-ID columns. I can claim shared *repository nomenclature coverage* for these names. I cannot claim matched concentration, matched annotation certainty, pathway enrichment, or a common biological response across the studies.”

### L4.4 Common pitfall and repair

**Pitfall:** collapsing duplicate RefMet names across analysis modes before checking their analytical origin—or selecting whichever duplicate gives the desired pattern.

**Repair:** keep `analysis_feature_id`, reported name, analysis ID, and RefMet name together. Select an analysis prospectively for PCA. If a later analysis collapses modes, specify and justify one deterministic rule, evaluate sensitivity, and preserve the pre-collapse audit table.

### Optional extension

For a statistically stronger next step, choose a small set of well-audited features within one study and estimate participant-centered change from baseline using a repeated-measures or mixed-effects model. Handle FARMM's imbalance explicitly and include the antibiotic/PEG period in the estimand. Repeat independently in ST003348, then compare effect *direction and uncertainty* rather than raw peak-area magnitude.

A second extension is to replace the conservative name-collision rule with verified assay documentation. That requires evidence beyond the study-level REST table; record every decision in the mapping audit rather than silently restoring a label.

**Lesson 4 complete when:** both PCAs use separate study/analysis matrices, the expected sample/feature summaries appear, and your four-sentence interpretation states observation, context, alternatives, and boundary without a causal cross-study claim.

<a id="nb-l5"></a>
## Lesson 5 - Access patterns and transfer (`NB-L5`)

**Guide cross-reference.** Read learner-guide Lesson 5 and open the access-pattern transfer checklist. This section does not retrieve governed data. Instead:

1. Review which inputs came from live endpoints versus the validated public cache.
2. Choose a target dataset/release and intended action.
3. Record dated first-party access evidence outside the notebook.
4. Edit the transfer-decision scaffold and select `GO`, `REVISE`, `WAIT`, or `STOP`.
5. Finish with the reproducibility audit, then return to the guide for the posttest.

### L5.1 Learner edit - draft a transfer decision

The defaults describe this public cached tutorial. Replace them with evidence for your exact target. Do not place credentials, governed data, signed URLs, or personal information in this notebook.

In [17]:
# Learner-edit cell: replace every value with dated evidence for your target resource.
TARGET_RESOURCE = "Metabolomics Workbench ST001521/ST003348 public release"
INTENDED_ACTION = "Run the cached educational workflow locally"
ACCESS_EVIDENCE_DATE = "2026-08-14"
INPUT_BOUNDARY = "Public split REST responses or versioned public cache"
COMPUTE_LOCATION = "Local learner environment"
PERMITTED_OUTPUT = "Aggregate tables, figures, code, and provenance records"
UNRESOLVED_REQUIREMENTS = []
DECISION = "GO"  # Choose GO, REVISE, WAIT, or STOP after reviewing current evidence.

if DECISION not in {"GO", "REVISE", "WAIT", "STOP"}:
    raise ValueError("DECISION must be GO, REVISE, WAIT, or STOP")

transfer_decision = pd.DataFrame(
    [
        {
            "target_resource": TARGET_RESOURCE,
            "intended_action": INTENDED_ACTION,
            "evidence_date": ACCESS_EVIDENCE_DATE,
            "input_boundary": INPUT_BOUNDARY,
            "compute_location": COMPUTE_LOCATION,
            "permitted_output": PERMITTED_OUTPUT,
            "unresolved_requirements": "; ".join(UNRESOLVED_REQUIREMENTS) or "none recorded",
            "decision": DECISION,
        }
    ]
)
display(transfer_decision.T)

,0
target_resource,Metabolomics Workbench ST001521/ST003348 public release
intended_action,Run the cached educational workflow locally
evidence_date,2026-08-14
input_boundary,Public split REST responses or versioned public cache
compute_location,Local learner environment
permitted_output,"Aggregate tables, figures, code, and provenance records"
unresolved_requirements,none recorded
decision,GO


### L5.2 Run now - final reproducibility audit (`NB-REPRO`)

The final cell inventories every tutorial-generated CSV and PNG with its byte size and SHA-256 checksum. It also asserts that the expected counts and figures exist and that no SVG artifact was produced.

In [18]:
expected_csvs = {
    "input_source_log.csv",
    "tidy_factors.csv",
    "metabolite_mapping_audit.csv",
    "refmet_overlap_audit.csv",
    "metabolite_crosswalk.csv",
    "refmet_class_summary.csv",
    "refmet_overlap_counts.csv",
    "ST001521_AN002534_pca_scores.csv",
    "ST001521_AN002534_pca_loadings.csv",
    "ST001521_AN002534_pca_feature_qc.csv",
    "ST003348_AN005483_pca_scores.csv",
    "ST003348_AN005483_pca_loadings.csv",
    "ST003348_AN005483_pca_feature_qc.csv",
    "pca_preprocessing_summary.csv",
}
expected_pngs = {
    "refmet_overlap_summary.png",
    "refmet_class_summary.png",
    "ST001521_AN002534_pca.png",
    "ST003348_AN005483_pca.png",
}

assert expected_csvs.issubset({path.name for path in DERIVED_DIR.glob("*.csv")})
assert expected_pngs.issubset({path.name for path in FIGURES_DIR.glob("*.png")})
assert not list(FIGURES_DIR.glob("*.svg"))
assert overlap_counts["raw_exact_refmet_overlap"] == 153
assert overlap_counts["conservative_biological_refmet_overlap"] == 145

artifact_paths = sorted(
    [DERIVED_DIR / name for name in expected_csvs]
    + [FIGURES_DIR / name for name in expected_pngs]
)
manifest = pd.DataFrame(
    [
        {
            "artifact": str(path.relative_to(MODULE_DIR)),
            "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
        for path in artifact_paths
    ]
)
manifest_path = DERIVED_DIR / "tutorial_output_manifest.csv"
manifest.to_csv(manifest_path, index=False)
display(manifest)
print("Tutorial completed: 226 biological samples, 145 conservative shared RefMet names, two separate PCAs.")

,artifact,bytes,sha256
0,data/derived/ST001521_AN002534_pca_feature_qc.csv,24698,b11aaa0c2df0fb6096b97df4ea1575eac9010b13b11619fc904225a7b71ab4b6
1,data/derived/ST001521_AN002534_pca_loadings.csv,21222,1d94d06e5588613e94b0c8d91f494f9295331db9a41af754c2eab752d70624f6
2,data/derived/ST001521_AN002534_pca_scores.csv,49521,980ac36ed7b9244b40332359e2518c476625c9b4d7c31b83d53db1000a4c714f
3,data/derived/ST003348_AN005483_pca_feature_qc.csv,37303,b4706017df58a1a1a85469b13ba474fd455a5b34ef0565a0307543de53b7095c
4,data/derived/ST003348_AN005483_pca_loadings.csv,34138,b15b8da317f224cddc1d89c4f217b49526fe102b21230b2abe963622430d2188
5,data/derived/ST003348_AN005483_pca_scores.csv,22923,17371470eb856f06c8b36f374ca26cfac53285faf185381993a703de9b99b6a8
6,data/derived/input_source_log.csv,1918,6de664b68cf28f2b077f9374727c22cfc8edde4b352adc9fff4413faf88181d6
7,data/derived/metabolite_crosswalk.csv,68152,38dd1f467c4ab5b8aab47ad82ffe817d72121cd67a657b5429583a76f473a44a
8,data/derived/metabolite_mapping_audit.csv,524175,6853b4b4c7bb4c8db8fd5fab68038b85489a3b66190f92e8598a016d8a3353c2
9,data/derived/pca_preprocessing_summary.csv,480,bd7c0c99a619b77075716d75375d96c869291bd0101a8e8716ad2a84dc788f81


Tutorial completed: 226 biological samples, 145 conservative shared RefMet names, two separate PCAs.


**Lesson 5 and notebook complete when:** the artifact manifest is displayed without errors, your transfer checklist cites dated first-party evidence, and your decision states any unresolved requirements. Save the notebook, return to the learner guide, and complete the posttest.